# CoMETs Results
Loads all completed runs from `runs/latent/` and `runs/unit/`. Re-run any cell to refresh.

In [ ]:
import json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.ndimage import gaussian_filter1d
from scipy.stats import spearmanr
from sklearn.decomposition import PCA as skPCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

import sys
for _p in [Path('.'), *Path('.').resolve().parents]:
    if (_p / 'comets' / '__init__.py').exists():
        if str(_p) not in sys.path: sys.path.insert(0, str(_p))
        break
from comets import train_unit, SamplingConfig, UnitTrainingConfig
from comets.unit_eval import ratemap_correlation

ROOT       = Path('.').resolve()
LATENT_DIR = ROOT / 'runs' / 'latent'
UNIT_DIR   = ROOT / 'runs' / 'unit'
DATA_DIR   = '/mnt/d/comets/data/crcns_mouse'

# ec013.157 excluded from main analysis: animal stays on one side at trial end,
# severely distorting the position distribution and collapsing spatial decoding R².
SESSIONS    = ['ec013.156', 'ec013.440', 'ec013.441', 'ec013.442']
SESSION_BAD = 'ec013.157'

PROJ_DIM = 8;  D_U = 8

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

def _style_map(styler, fn, **kw):
    try: return styler.map(fn, **kw)
    except AttributeError: return styler.applymap(fn, **kw)

print('Setup OK')

---
## 1 · Load all runs

In [ ]:
def load_latent_runs(sessions=SESSIONS):
    rows = []
    for run_dir in sorted(LATENT_DIR.glob('*')):
        cfg_p = run_dir / 'config.json'
        if not cfg_p.exists(): continue
        cfg = json.load(open(cfg_p))
        for sd in sorted(run_dir.glob('ec013.*')):
            if sd.name not in sessions: continue
            lcsv = sd / 'latent_log.csv'
            if not lcsv.exists(): continue
            log = pd.read_csv(lcsv)
            rows.append(dict(
                run_name   = cfg['run_name'],
                encoder    = cfg['encoder_name'],
                loss       = cfg['loss_name'],
                session    = sd.name,
                best_r2    = float(log['r2_concat'].max()),
                last_r2    = float(log['r2_concat'].iloc[-1]),
                best_epoch = int(log.loc[log['r2_concat'].idxmax(),'epoch']),
                n_epochs   = int(log['epoch'].max()),
                log        = log,
                sess_dir   = sd,
            ))
    return pd.DataFrame(rows)

def load_unit_runs(sessions=SESSIONS):
    rows = []
    for run_dir in sorted(UNIT_DIR.glob('*')):
        cfg_p = run_dir / 'config.json'
        if not cfg_p.exists(): continue
        cfg = json.load(open(cfg_p))
        for sd in sorted(run_dir.glob('ec013.*')):
            if sd.name not in sessions: continue
            lcsv = sd / 'unit_log.csv'
            if not lcsv.exists(): continue
            log = pd.read_csv(lcsv)
            rows.append(dict(
                run_name     = cfg['run_name'],
                latent_run   = cfg['latent_run_name'],
                latent_ckp   = cfg['latent_ckp'],
                lambda_coact = float(cfg['lambda_coact']),
                lambda_ctx   = float(cfg['lambda_ctx']),
                session      = sd.name,
                best_rho     = float(log['rho_ratemap'].max()),
                last_rho     = float(log['rho_ratemap'].iloc[-1]),
                best_epoch   = int(log.loc[log['rho_ratemap'].idxmax(),'epoch']),
                n_epochs     = int(log['epoch'].max()),
                log          = log,
            ))
    return pd.DataFrame(rows)

lat = load_latent_runs()
unt = load_unit_runs()
print(f'Latent: {lat["run_name"].nunique()} configs × {lat["session"].nunique()} sessions = {len(lat)} rows')
print(f'Unit  : {unt["run_name"].nunique() if len(unt) else 0} configs = {len(unt)} rows')
lat[['run_name','session','n_epochs','best_r2']].sort_values(['run_name','session'])

---
## 1b · ec013.157 — excluded session (position bias illustration)

This session is excluded from all analyses. The animal stayed on one side of the track at the end of the trial, creating a heavily skewed position distribution that collapses spatial decoding R² to strongly negative values regardless of model configuration.

In [ ]:
lat_bad = load_latent_runs(sessions=[SESSION_BAD])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

data_bad = pickle.load(open(f'{DATA_DIR}/{SESSION_BAD}/{SESSION_BAD}_lfp_spike.pkl', 'rb'))
pos_bad  = np.asarray(data_bad['position_50hz'], dtype=np.float64)
if pos_bad.ndim == 2: pos_bad = pos_bad[:, 0]
axes[0].hist(pos_bad, bins=60, color='salmon', edgecolor='none', alpha=0.85)
axes[0].set(xlabel='Position (a.u.)', ylabel='Time bins',
            title=f'{SESSION_BAD} — position distribution\n(skewed by end-of-trial parking)')
axes[0].grid(alpha=0.3)

clrs = plt.cm.tab10(np.linspace(0, 1, lat_bad['run_name'].nunique()))
clr_map = {r: c for r, c in zip(sorted(lat_bad['run_name'].unique()), clrs)}
for _, row in lat_bad.iterrows():
    axes[1].plot(row['log']['epoch'], row['log']['r2_concat'],
                 color=clr_map[row['run_name']], lw=1.2, alpha=0.8,
                 label=row['run_name'])
axes[1].axhline(0, color='k', lw=0.8, ls='--')
axes[1].set(xlabel='Epoch', ylabel='R² concat (10-fold CV)',
            title=f'{SESSION_BAD} — latent spatial R² (all configs fail)')
axes[1].legend(fontsize=6, ncol=2); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f'Position range: [{pos_bad.min():.1f}, {pos_bad.max():.1f}]  '
      f'median={np.median(pos_bad):.1f}  '
      f'% time at upper 10%: {(pos_bad > np.percentile(pos_bad,90)).mean()*100:.0f}%')

---
## 2 · Latent learning curves

In [ ]:
if len(lat):
    runs  = sorted(lat['run_name'].unique())
    sess  = sorted(lat['session'].unique())
    clrs  = {r: cm.tab10(i/max(len(runs)-1,1)) for i,r in enumerate(runs)}
    fig, axes = plt.subplots(1, len(sess), figsize=(4*len(sess), 3.5), sharey=True)
    if len(sess)==1: axes=[axes]
    for ax, s in zip(axes, sess):
        for _, row in lat[lat['session']==s].iterrows():
            ax.plot(row['log']['epoch'], row['log']['r2_concat'],
                    color=clrs[row['run_name']], lw=1.4,
                    label=row['run_name'] if s==sess[0] else '_')
            ax.axvline(row['best_epoch'], color=clrs[row['run_name']], lw=0.6, ls='--', alpha=0.4)
        ax.set(title=s, xlabel='Epoch'); ax.grid(alpha=0.3)
    axes[0].set_ylabel('R² concat (10-fold CV)')
    fig.legend(*axes[0].get_legend_handles_labels(), loc='lower center',
               ncol=3, bbox_to_anchor=(0.5,-0.14), fontsize=8)
    fig.suptitle('Latent spatial decoding R² over training', fontweight='bold')
    plt.tight_layout(); plt.show()

---
## 3 · Per-session best latent — grouped stats

For each session: rank all latent configs by best R². Shows which encoder/loss wins per session and how consistent it is across animals.

In [ ]:
if len(lat):
    pivot = lat.pivot_table(index='run_name', columns='session',
                             values='best_r2', aggfunc='max')
    pivot['mean'] = pivot.mean(axis=1)
    pivot['std']  = pivot.std(axis=1)
    pivot = pivot.sort_values('mean', ascending=False)

    vmin = pivot.drop(columns=['mean','std']).min().min()
    vmax = pivot.drop(columns=['mean','std']).max().max()
    normed = lambda v: (v - vmin) / max(vmax - vmin, 1e-6)

    def heat(val):
        if pd.isna(val): return ''
        g = normed(val)
        return f'background-color: rgba(0,{int(180*g)},{int(220*(1-g))},{0.25+0.55*g:.2f})'

    display(_style_map(pivot.style.format('{:.3f}'), heat)
            .set_caption('Best concat R² per session (low → high)'))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    df_bar = pivot[['mean','std']].reset_index().sort_values('mean', ascending=True)
    clrs_enc = {'balanced':'tab:blue','heavy_spike':'tab:orange','heavy_lfp':'tab:green'}
    bar_clrs = [clrs_enc.get(r.split('_')[0], 'gray') for r in df_bar['run_name']]
    axes[0].barh(df_bar['run_name'], df_bar['mean'], xerr=df_bar['std'],
                 color=bar_clrs, capsize=3, alpha=0.8)
    axes[0].set(xlabel='Mean best R² (across sessions)', title='Latent configs ranked')
    axes[0].axvline(df_bar['mean'].max(), color='red', lw=0.8, ls='--')
    axes[0].grid(axis='x', alpha=0.3)

    winner_per_sess = {}
    for s in lat['session'].unique():
        sub = lat[lat['session']==s].sort_values('best_r2', ascending=False)
        if len(sub): winner_per_sess[s] = sub.iloc[0]['run_name']
    winner_counts = pd.Series(winner_per_sess).value_counts()
    axes[1].bar(winner_counts.index, winner_counts.values, alpha=0.8, color='steelblue')
    axes[1].set(ylabel='# sessions where config ranks #1',
                title='Which latent config wins most often?')
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout(); plt.show()
    print('Overall winner (most sessions):', winner_counts.idxmax())
    print('Overall winner (mean R²):      ', pivot['mean'].idxmax())

---
## 4 · PCA baseline for unit embeddings

Unit encoder trained on PCA latents — the floor for rho_ratemap. Results are cached to `runs/pca_baseline/` so subsequent runs load instantly.

In [ ]:
PCA_CACHE_DIR = ROOT / 'runs' / 'pca_baseline'

def load_session(session):
    data = pickle.load(open(f'{DATA_DIR}/{session}/{session}_lfp_spike.pkl','rb'))
    sc   = np.asarray(data['spike_bin'],      dtype=np.float32)
    sr   = np.asarray(data['spike_rate'],     dtype=np.float32)
    pos  = np.asarray(data['position_50hz'],  dtype=np.float64)
    lfp  = np.asarray(data['lfp_theta_50hz'], dtype=np.float32)
    fs   = float(data['fs_common'])
    if pos.ndim==2: pos=pos[:,0]
    rmin,rmax = sr.min(0,keepdims=True), sr.max(0,keepdims=True)
    return sc, (sr-rmin)/(rmax-rmin+1e-9), lfp, pos, fs

def compute_ratemaps(spike_counts, position, fs, n_bins=40, sigma=2.0):
    edges    = np.linspace(position.min(), position.max(), n_bins+1)
    pos_bins = np.clip(np.digitize(position, edges)-1, 0, n_bins-1)
    occ      = np.bincount(pos_bins, minlength=n_bins) / fs
    rm = np.zeros((spike_counts.shape[1], n_bins), dtype=np.float32)
    for i in range(spike_counts.shape[1]):
        rm[i] = np.bincount(pos_bins, weights=spike_counts[:,i], minlength=n_bins) / (occ+1e-9)
    return gaussian_filter1d(rm, sigma=sigma, axis=1)

PCA_BASELINE = {}
RATEMAPS     = {}

for sess in SESSIONS:
    sc, snorm, lfp, pos, fs = load_session(sess)
    rm = compute_ratemaps(sc, pos, fs)
    RATEMAPS[sess] = rm

    cache_dir = PCA_CACHE_DIR / sess
    e_path    = cache_dir / 'E_pca.npy'
    rho_path  = cache_dir / 'rho.json'

    if e_path.exists() and rho_path.exists():
        rho = json.load(open(rho_path))['rho']
        print(f'{sess}  PCA rho = {rho:+.4f}  [loaded from cache]')
    else:
        Z_spk = skPCA(n_components=PROJ_DIM, random_state=0).fit_transform(snorm).astype(np.float32)
        Z_lf  = skPCA(n_components=PROJ_DIM, random_state=0).fit_transform(lfp).astype(np.float32)
        enc   = train_unit(None, sc, SamplingConfig(batch_size=256),
                           UnitTrainingConfig(epochs=200, d_u=D_U, lambda_coact=0.5, lambda_ctx=0.5),
                           z_spikes=Z_spk, z_lfp=Z_lf)
        E   = enc.embeddings().numpy()
        rho, _ = ratemap_correlation(E, rm)
        cache_dir.mkdir(parents=True, exist_ok=True)
        np.save(e_path, E)
        json.dump({'rho': float(rho)}, open(rho_path, 'w'))
        print(f'{sess}  PCA rho = {rho:+.4f}  [trained & cached]')

    PCA_BASELINE[sess] = rho

print(f'\nPCA mean rho = {np.mean(list(PCA_BASELINE.values())):+.4f}')

---
## 5 · Co-activity vs context landscape

For each unit config (lambda_coact vs lambda_ctx), what is the average rho_ratemap across sessions and latent setups? Reveals whether co-activity or context dominates.

In [ ]:
if len(unt) == 0:
    print('No unit runs yet.')
else:
    grp = unt.groupby(['lambda_coact','lambda_ctx'])['best_rho'].agg(['mean','std','count']).reset_index()
    grp['label'] = grp.apply(lambda r: f"lc={r.lambda_coact:.2f}\nlx={r.lambda_ctx:.2f}", axis=1)
    pca_mean = np.mean(list(PCA_BASELINE.values())) if PCA_BASELINE else 0

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    grp_s = grp.sort_values('mean', ascending=False)
    clrs  = ['green' if v > pca_mean else 'salmon' for v in grp_s['mean']]
    axes[0].bar(grp_s['label'], grp_s['mean'], yerr=grp_s['std'],
                color=clrs, capsize=4, alpha=0.85)
    axes[0].axhline(pca_mean, color='gray', ls='--', lw=1.5, label=f'PCA baseline ({pca_mean:.3f})')
    axes[0].set(ylabel='Mean best rho_ratemap',
                title='Unit loss config effect (averaged over all latent runs & sessions)')
    axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

    sess_grp = unt.groupby(['lambda_coact','lambda_ctx','session'])['best_rho'].max().reset_index()
    pivot_u = sess_grp.pivot_table(index=['lambda_coact','lambda_ctx'], columns='session',
                                    values='best_rho')
    pivot_u['mean'] = pivot_u.mean(axis=1)
    pivot_u = pivot_u.sort_values('mean', ascending=False)

    im = axes[1].imshow(pivot_u.drop(columns='mean').values, aspect='auto', cmap='RdYlGn')
    axes[1].set_xticks(range(len(pivot_u.columns)-1))
    axes[1].set_xticklabels([c for c in pivot_u.columns if c!='mean'], rotation=30, fontsize=8)
    axes[1].set_yticks(range(len(pivot_u)))
    axes[1].set_yticklabels([f'lc={lc:.2f} lx={lx:.2f}'
                              for lc,lx in pivot_u.index], fontsize=8)
    axes[1].set_title('Rho per session (unit config × session heatmap)')
    plt.colorbar(im, ax=axes[1], label='rho_ratemap')

    plt.tight_layout(); plt.show()

    print('Best unit loss config (mean rho):')
    for (lc,lx), row in grp_s.set_index(['lambda_coact','lambda_ctx']).iterrows():
        marker = ' ← best' if lc==grp_s.iloc[0]['lambda_coact'] and lx==grp_s.iloc[0]['lambda_ctx'] else ''
        print(f'  lc={lc:.2f}  lx={lx:.2f}  mean={row["mean"]:+.4f}  Δ PCA={row["mean"]-pca_mean:+.4f}{marker}')

---
## 6 · Does the best latent R² predict the best unit rho?

Key question: is the latent config that decodes space best also the one that produces the best unit embeddings?

In [ ]:
if len(unt) == 0:
    print('No unit runs yet.')
else:
    lat_lookup = lat.set_index(['run_name','session'])['best_r2']
    unt2 = unt.copy()
    unt2['latent_r2'] = unt2.apply(
        lambda r: lat_lookup.get((r['latent_run'], r['session']), np.nan), axis=1)
    unt2 = unt2.dropna(subset=['latent_r2'])

    if len(unt2) == 0:
        print('No matched latent+unit rows yet.')
    else:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        sess_colors = {s: cm.tab10(i/max(len(SESSIONS)-1,1)) for i,s in enumerate(sorted(SESSIONS))}
        for sess, sub in unt2.groupby('session'):
            axes[0].scatter(sub['latent_r2'], sub['best_rho'], s=20, alpha=0.5,
                            color=sess_colors[sess], label=sess)
        rho_all, p_all = spearmanr(unt2['latent_r2'], unt2['best_rho'])
        axes[0].set(xlabel='Latent R² (spatial decoding)', ylabel='Unit rho_ratemap',
                    title=f'Latent R² vs Unit ρ\nSpearman ρ={rho_all:+.3f}  p={p_all:.3f}')
        axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)

        sess_list, rho_list = [], []
        for sess, sub in unt2.groupby('session'):
            if len(sub) >= 3:
                r, _ = spearmanr(sub['latent_r2'], sub['best_rho'])
                sess_list.append(sess); rho_list.append(r)
        axes[1].bar(sess_list, rho_list,
                    color=['green' if v>0 else 'salmon' for v in rho_list], alpha=0.85)
        axes[1].axhline(0, color='k', lw=0.8)
        axes[1].set(ylabel='Spearman ρ', ylim=(-1,1),
                    title='Rank correlation (latent R² → unit ρ)\nper session')
        axes[1].tick_params(axis='x', rotation=25); axes[1].grid(axis='y', alpha=0.3)

        agreement = []
        for sess, sub in unt2.groupby('session'):
            best_lat_run  = sub.loc[sub['latent_r2'].idxmax(), 'latent_run']
            best_unit_run = sub.loc[sub['best_rho'].idxmax(),  'latent_run']
            agreement.append({'session': sess, 'best_lat': best_lat_run,
                               'best_unit': best_unit_run,
                               'agree': best_lat_run == best_unit_run})
        agree_df = pd.DataFrame(agreement)
        n_agree = agree_df['agree'].sum()
        axes[2].bar(['Agree (best lat = best unit)', 'Disagree'],
                    [n_agree, len(agree_df)-n_agree],
                    color=['steelblue','salmon'], alpha=0.85)
        axes[2].set(title=f'Does rank-1 latent = rank-1 unit?\n({n_agree}/{len(agree_df)} sessions agree)',
                    ylabel='# sessions')
        axes[2].grid(axis='y', alpha=0.3)

        plt.tight_layout(); plt.show()

        print('\nPer-session agreement:')
        for _, row in agree_df.iterrows():
            mark = '✓' if row['agree'] else '✗'
            print(f'  {mark} {row["session"]:12s}  best_latent={row["best_lat"]:35s}  best_unit={row["best_unit"]}')

---
## 7 · Unit summary table vs PCA baseline

In [ ]:
if len(unt):
    pivot = unt.pivot_table(index='run_name', columns='session',
                             values='best_rho', aggfunc='max')
    pivot['mean_ρ'] = pivot.mean(axis=1)
    pivot = pivot.sort_values('mean_ρ', ascending=False)
    pca_row = {s: PCA_BASELINE.get(s, np.nan) for s in pivot.columns if s!='mean_ρ'}
    pca_row['mean_ρ'] = np.nanmean(list(pca_row.values()))
    pivot.loc['─── PCA baseline ───'] = pca_row

    def color_vs_pca(val, col):
        if pd.isna(val): return ''
        base = PCA_BASELINE.get(col, np.nan)
        if pd.isna(base): return ''
        d = val - base
        if d > 0: return f'background-color: rgba(0,180,0,{min(d*8,0.7):.2f})'
        return      f'background-color: rgba(200,0,0,{min(-d*8,0.5):.2f})'

    s = pivot.style.format('{:.3f}')
    for col in [c for c in pivot.columns if c != 'mean_ρ']:
        s = _style_map(s, lambda v, c=col: color_vs_pca(v, c), subset=[col])
    display(s.set_caption('Best rho_ratemap — green = above PCA, red = below'))
    print('Best unit config:', pivot['mean_ρ'].drop('─── PCA baseline ───').idxmax())

---
## 8 · Best checkpoint vs Last checkpoint

In [ ]:
if len(unt) and unt['latent_ckp'].nunique() == 2:
    best_r = unt[unt['latent_ckp']=='best']
    last_r = unt[unt['latent_ckp']=='last']
    merged = best_r[['latent_run','lambda_coact','lambda_ctx','session','best_rho']].merge(
        last_r[['latent_run','lambda_coact','lambda_ctx','session','best_rho']],
        on=['latent_run','lambda_coact','lambda_ctx','session'], suffixes=('_best','_last'))

    fig, ax = plt.subplots(figsize=(5,5))
    lo = merged[['best_rho_best','best_rho_last']].min().min()-0.02
    hi = merged[['best_rho_best','best_rho_last']].max().max()+0.02
    ax.plot([lo,hi],[lo,hi],'k--',lw=0.8,label='equal')
    ax.scatter(merged['best_rho_last'], merged['best_rho_best'], alpha=0.5, s=25)
    n_win = (merged['best_rho_best'] > merged['best_rho_last']).sum()
    ax.set(xlabel='rho  (last-epoch latent)', ylabel='rho  (best-R² latent)',
           title=f'Best vs Last ckp  —  {n_win}/{len(merged)} cases best-ckp wins',
           xlim=[lo,hi], ylim=[lo,hi])
    ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    print(f'Mean Δrho (best−last): {(merged["best_rho_best"]-merged["best_rho_last"]).mean():+.4f}')
else:
    print('Need both best and last checkpoint runs to compare.')

---
## 9 · Recommended configuration

In [ ]:
print('═'*60)
print('LATENT — top 3 configs by mean R²')
print('═'*60)
if len(lat):
    for name, val in lat.groupby('run_name')['best_r2'].mean().sort_values(ascending=False).head(3).items():
        print(f'  {name:<45}  R² = {val:.4f}')

pca_mean = np.mean(list(PCA_BASELINE.values())) if PCA_BASELINE else 0
print()
print('═'*60)
print(f'UNIT — configs above PCA baseline ({pca_mean:.4f})')
print('═'*60)
if len(unt):
    for name, val in unt.groupby('run_name')['best_rho'].mean().sort_values(ascending=False).items():
        above = val > pca_mean
        marker = f'  Δ PCA = {val-pca_mean:+.4f}' + (' ✓' if above else ' ✗')
        if above: print(f'  {name:<65}  rho = {val:.4f}{marker}')